# Retrieval versus Long Context

*Attention complexity, the limits of "more context," and positional bias — read as a rate–distortion choice at the generation boundary.*

This notebook imports the canonical module `retrieval_vs_long_context.py` (which owns every number) and
walks the topic section by section. It never redefines the math; it calls into the module.

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd()))
import numpy as np
import retrieval_vs_long_context as rvlc

corpus = rvlc._corpus()
print(f"thick finance corpus: {corpus['K']} companies (= answers), {corpus['R']} relevant passages each, "
      f"{corpus['n_passages']} passages, {corpus['n_queries']} queries, d={corpus['dim']}")

## 1 · Attention is quadratic — the rate axis

Full self-attention over a context of $k$ passages of $L$ tokens forms a $kL \times kL$ score matrix:
the arithmetic cost is $\Theta((kL)^2)$. Doubling the context quadruples the compute. This is the *rate*
the long-context option pays. (FlashAttention lowers the *memory* to $O(n)$ but leaves this FLOP count
unchanged.)

In [ ]:
for k in (1, 4, 8, 16):
    print(f"  k={k:2d}  cost=(k·L)^2 = {int(rvlc.attention_cost(k)):>10,d} FLOP-units")
print("doubling k from 4 to 8 multiplies cost by", rvlc.attention_cost(8) / rvlc.attention_cost(4))

## 2 · More context is not better

We retrieve the top-$k$ passages and read them under a **finite attention budget** (softmax weights that
sum to one). Answer quality $Q(k)$ is the posterior mass the generator places on the *true* company. Even
though the answer is reliably retrieved (recall@1 ≈ 1), $Q$ is highest at the **smallest** context and
declines as $k$ grows. The answer **entropy** $H$ — the distortion, in bits — rises in lockstep.

In [ ]:
rows = rvlc.quality_recall_precision_cost(corpus)
print(f"{'k':>2}  {'Q':>6}  {'H(bits)':>7}  {'recall_R':>8}  {'prec':>6}  {'util':>6}")
for r in rows:
    print(f"{r['k']:>2}  {r['Q']:.3f}  {r['H']:>7.3f}  {r['recall']:>8.3f}  {r['precision']:.3f}  {r['util']:.3f}")
opt = rvlc.find_optimum(rows)
print(f"\noptimum at k={opt['k_star']}; Q(1)={opt['Q_1']:.3f} -> Q(n)={opt['Q_n']:.3f} "
      f"(stuffing costs {opt['Q_1']-opt['Q_n']:.3f} of quality)")

Two mechanisms drive the decline. While the top-$k$ is *all relevant* (precision ≈ 1, small $k$), the
extra passages are **redundant** — a second filing of the same company barely moves the belief. We import
the previous topic's `saturation_table` to re-derive this on the same geometry: a redundant document adds
far less than the first, and less than a genuinely novel one.

In [ ]:
sat = rvlc.saturation_table(rvlc._proto_view_corpus(corpus), rvlc.TAU, rvlc.TAU_DOC)
print(f"standalone (first filing): {sat['standalone']:.3f} bits")
print(f"redundant  (same again):   {sat['redundant']:.3f} bits   <-- diminishing returns")
print(f"novel      (different):    {sat['novel']:.3f} bits")

Once the relevant set is exhausted, the top-$k$ admits **same-sector distractors** (confusable, cosine
≈ 0.6). They draw attention budget away from the relevant passages and pull the posterior toward the wrong
company — precision and utilization fall, entropy rises, $Q$ declines.

### The right amount of context depends on retrieval quality

When retrieval is *good*, the optimum is the smallest covering context. When retrieval is *hard* (here a
much more ambiguous query, recall@1 ≈ 0.47), too few passages risks **missing** the answer — a small
interior optimum appears, the classic recall–precision balance. Better retrieval always wins at the peak,
and stuffing the window is dominated either way.

In [ ]:
fam = rvlc.retrieval_quality_family(corpus)
print("k :  good_Q  hard_Q")
for k, g, h in zip(rvlc.K_GRID, fam['good_Q'], fam['hard_Q']):
    print(f"{k:>2}:  {g:.3f}   {h:.3f}")
print(f"\npeak quality — good: {fam['good_peak']:.3f}   hard: {fam['hard_peak']:.3f}")

## 3 · Lost in the middle — positional bias is a soft erasure

A relevant passage buried in the *middle* of a long context is read with attenuated attention (Liu et al.,
2023). We model this positional weight as a U-shaped multiplier — high at the ends, depressed in the
center — and slide the gold passage across all positions. A buried passage behaves like a **partial
erasure**: its quality is lowest at the center.

In [ ]:
pq = rvlc.positional_quality(corpus)
qs = [r['Q'] for r in pq]
print("position :", " ".join(f"{r['pos']:>5}" for r in pq))
print("Q        :", " ".join(f"{r['Q']:.3f}" for r in pq))
mid = len(qs) // 2
print(f"\nends Q≈{qs[0]:.3f}/{qs[-1]:.3f}  vs  middle Q≈{qs[mid]:.3f}  (buried = worst)")

## 4 · The rate–distortion verdict

Plot quality against cost. The focused-retrieval point sits up and to the left; stuffing the window pays
quadratically more compute for strictly worse answers. Retrieve the few passages that carry the answer —
the gate to *context selection* (choosing a small, diverse, high-coverage subset).

In [ ]:
opt = rvlc.find_optimum(rows)
star, last = rows[opt['i_star']], rows[-1]
print(f"optimum k={star['k']:>2}: cost={int(star['cost']):>10,d}  Q={star['Q']:.3f}")
print(f"stuff   k={last['k']:>2}: cost={int(last['cost']):>10,d}  Q={last['Q']:.3f}")
print(f"\nstuffing costs {last['cost']/star['cost']:.0f}x the compute for {star['Q']-last['Q']:.3f} LESS quality")

## 5 · Every claim is a test

The module asserts each pedagogical claim. Re-run them here; this cell must exit cleanly.

In [ ]:
rvlc._run_all()
print("\nall checks passed.")